Model Comparisons for Predicting Campaign Acceptance

In [3]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

In [4]:
RANDOM_STATE = 1234

In [5]:
PROJECT_ROOT = Path.cwd().parent   # assumes notebook lives in notebooks/, project root is one level up
DATA_DIR = PROJECT_ROOT / "data" / "processed"
RAW_DATA_PATH = DATA_DIR / "customer_personality_cleaned.csv"

In [6]:
df = pd.read_csv(RAW_DATA_PATH)

In [7]:
df.columns

Index(['Education', 'Marital_Status', 'Income', 'MntWines', 'MntFruits',
       'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts',
       'MntGoldProds', 'NumDealsPurchases', 'NumWebPurchases',
       'NumCatalogPurchases', 'NumStorePurchases', 'NumWebVisitsMonth',
       'Complain', 'age', 'Accepted_Cmp', 'duration', 'children',
       'HighestPurchaseSource'],
      dtype='object')

In [8]:
target_col = df['Accepted_Cmp']

In [9]:
feature_cols = df.drop('Accepted_Cmp', axis=1)
categorical_cols = [c for c in feature_cols if df[c].dtype.name in ("category", "object")]
numeric_cols = [c for c in feature_cols if c not in categorical_cols]

In [15]:
preprocessor = ColumnTransformer(
    transformers= [
        ("num", StandardScaler(), numeric_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
    ]
)

### Creating a function to initialize each model I plan to use

In [11]:
def get_models() -> dict:
    return {
        "knn": KNeighborsClassifier(),
        "logistic_regression": LogisticRegression(max_iter=2000),
        "logistic_regression_l1": LogisticRegression(penalty="l1", solver="liblinear", max_iter=2000),
        "random_forest": RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE),
        "decision_tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
        "gradient_boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
    }


In [21]:
X = df.drop('Accepted_Cmp', axis=1)
y = (df['Accepted_Cmp'] == "Yes").astype(int)

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)

results = []

In [12]:
### Creating a function to compare each model

#### Evaluate Each Model

In [22]:
# --- KNN ---
knn = KNeighborsClassifier()
pipeline = Pipeline([("preprocess", preprocessor), ("model", knn)])
scores = cross_val_score(pipeline, X, y, cv=cv, scoring="roc_auc", n_jobs=-1)
results.append({"model": "knn", "mean_roc_auc": scores.mean(), "std_roc_auc": scores.std()})
print(f"knn: mean ROC AUC = {scores.mean():.4f} (+/- {scores.std():.4f})")


knn: mean ROC AUC = 0.7342 (+/- 0.0327)


In [23]:
# --- Logistic Regression ---
log_reg = LogisticRegression(max_iter=2000)
pipeline = Pipeline([("preprocess", preprocessor), ("model", log_reg)])
scores = cross_val_score(pipeline, X, y, cv=cv, scoring="roc_auc", n_jobs=-1)
results.append({"model": "logistic_regression", "mean_roc_auc": scores.mean(), "std_roc_auc": scores.std()})
print(f"logistic_regression: mean ROC AUC = {scores.mean():.4f} (+/- {scores.std():.4f})")


logistic_regression: mean ROC AUC = 0.7793 (+/- 0.0365)


In [24]:
# --- Logistic Regression (L1) ---
log_reg_l1 = LogisticRegression(penalty="l1", solver="liblinear", max_iter=2000)
pipeline = Pipeline([("preprocess", preprocessor), ("model", log_reg_l1)])
scores = cross_val_score(pipeline, X, y, cv=cv, scoring="roc_auc", n_jobs=-1)
results.append({"model": "logistic_regression_l1", "mean_roc_auc": scores.mean(), "std_roc_auc": scores.std()})
print(f"logistic_regression_l1: mean ROC AUC = {scores.mean():.4f} (+/- {scores.std():.4f})")


logistic_regression_l1: mean ROC AUC = 0.7797 (+/- 0.0366)


In [25]:
# --- Random Forest ---
rf = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE)
pipeline = Pipeline([("preprocess", preprocessor), ("model", rf)])
scores = cross_val_score(pipeline, X, y, cv=cv, scoring="roc_auc", n_jobs=-1)
results.append({"model": "random_forest", "mean_roc_auc": scores.mean(), "std_roc_auc": scores.std()})
print(f"random_forest: mean ROC AUC = {scores.mean():.4f} (+/- {scores.std():.4f})")


random_forest: mean ROC AUC = 0.8589 (+/- 0.0245)


In [26]:
# --- Decision Tree ---
dt = DecisionTreeClassifier(random_state=RANDOM_STATE)
pipeline = Pipeline([("preprocess", preprocessor), ("model", dt)])
scores = cross_val_score(pipeline, X, y, cv=cv, scoring="roc_auc", n_jobs=-1)
results.append({"model": "decision_tree", "mean_roc_auc": scores.mean(), "std_roc_auc": scores.std()})
print(f"decision_tree: mean ROC AUC = {scores.mean():.4f} (+/- {scores.std():.4f})")


decision_tree: mean ROC AUC = 0.7100 (+/- 0.0323)


In [27]:
# --- Gradient Boosting ---
gb = GradientBoostingClassifier(random_state=RANDOM_STATE)
pipeline = Pipeline([("preprocess", preprocessor), ("model", gb)])
scores = cross_val_score(pipeline, X, y, cv=cv, scoring="roc_auc", n_jobs=-1)
results.append({"model": "gradient_boosting", "mean_roc_auc": scores.mean(), "std_roc_auc": scores.std()})
print(f"gradient_boosting: mean ROC AUC = {scores.mean():.4f} (+/- {scores.std():.4f})")


gradient_boosting: mean ROC AUC = 0.8429 (+/- 0.0371)


In [28]:
comparison = pd.DataFrame(results).sort_values("mean_roc_auc", ascending=False).reset_index(drop=True)
comparison

,model,mean_roc_auc,std_roc_auc
0,random_forest,0.858917,0.024538
1,gradient_boosting,0.842875,0.037117
2,logistic_regression_l1,0.779684,0.036566
3,logistic_regression,0.779341,0.036487
4,knn,0.734170,0.032681
5,decision_tree,0.710016,0.032347
